In [1]:
perfil = "SRD1CIV"
#perfil = "CAN1CIV"
#perfil = "GDO1CIV"
#perfil = "STM1CIV"


tipo_processo = "EXECUÇÃO FISCAL"

In [2]:
# Importa tudo, loga, entra no perfil

#from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.support.select import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.alert import Alert
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import ElementClickInterceptedException

#Bibliotecas de Sistema
from datetime import datetime
from datetime import timedelta
from datetime import date
from datetime import timezone
import time
import re
import csv
import os
import requests
import psutil
from bs4 import BeautifulSoup
from eproc_driver import eproc as eproc
import sqlite3
from pathlib import Path
import io
import pandas as pd
from contextlib import closing
from sympy import false
from pydoc import text
from sympy import true
import PyPDF2

#bibliotecas de configuração
import pyotp
import configparser
import keyring

#bibliotecas de automação
import pyperclip
import pyautogui

#Bibliotecas de IA
from gemini import gemini as gemini
import ollama

#pasta_downloads = r"C:\Users\dodonin\Downloads"
pasta_downloads = r"D:\Douglas\Downloads"

navegador = eproc.novo_browser(pasta_downloads)

#configura variáveis
username = "dodonin"
password = keyring.get_password("eproc", username)
pyotop_code = "GJRGIYTCGBSGKYTEHE2TOZRUGFQTMMRQ"

#eproc.login_no_eproc_tj(navegador, username, password, pyotop_code)
eproc.login_no_eproc(navegador, username, password, pyotop_code)

# Entra no perfil da Vara
eproc.entrar_no_perfil(navegador, perfil)

d:\Douglas\Reps\UNICA\.venv\Lib\site-packages\seleniumwire\thirdparty\mitmproxy\contrib\kaitaistruct\tls_client_hello.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Driver do Eproc importado


KeyboardInterrupt: 

In [ ]:
#Define funções para pegar os dados da tabela

def pega_tabela_pagina(dados_tabela):
    tabela = navegador.find_element(By.ID, "tabelaLocalizadores")
    linhas = tabela.find_elements(By.TAG_NAME, "tr")[1:]  # Ignora o cabeçalho
    lastpage = false
    while lastpage == false:
        for linha in linhas:
            # Aguarda o carregamento do tbody da tabela antes de processar as linhas
            WebDriverWait(navegador, 10).until(
                EC.presence_of_element_located((By.XPATH, "//table[@id='tabelaLocalizadores']/tbody"))
            )
            colunas = linha.find_elements(By.TAG_NAME, "td")[:4]
            if len(colunas) >= 3:
                # Adiciona apenas a primeira linha, sem quebra de linha
                dados_tabela.append([colunas[1].text.split('\n')[0], colunas[2].text.split('\n')[0], colunas[3].text.split('\n')[0]])
            
            lastpage = true

def pega_ultima_peticao(navegador, num_processo):
    """
    Localiza a última petição ("PET") disponível no processo informado,
    captura a requisição real feita pelo navegador (Selenium Wire) para baixar o PDF
    e salva o arquivo localmente.

    Parâmetros:
        navegador     -> instância do Selenium Wire WebDriver já autenticada no eproc.
        num_processo  -> número do processo (será usado no nome do PDF).

    Retorna:
        (evento_id, caminho_pdf) se o download foi bem-sucedido.
        (None, None) caso não haja petição ou falha no download.
    """
    import os
    import requests
    from selenium.webdriver.common.by import By
    import glob

    # === 1. Coleta todos os eventos do processo ===
    documentos_eventos = []
    eventos = navegador.find_elements(By.CLASS_NAME, "td-evento")

    for evento in eventos:
        # ID do elemento que identifica o documento
        doc_id = evento.get_dom_attribute("id")

        # Sobe até a linha da tabela e pega o número do evento
        tr_element = evento.find_element(By.XPATH, "./ancestor::tr")
        evento_id = tr_element.find_element(By.XPATH, './td[2]').text
        evento_id = ''.join(filter(str.isdigit, evento_id))  # mantém só números

        # Verifica se é documento tipo PET (Petição)
        try:
            infra_link = evento.find_element(By.XPATH, ".//*[contains(@class, 'infraLinkDoc')]")
            data_nome = infra_link.get_attribute("data-nome")
        except Exception:
            data_nome = None

        documentos_eventos.append((doc_id, data_nome, evento_id))

    # === 2. Filtra apenas documentos do tipo PET ===
    documentos_eventos_filtrados = [item for item in documentos_eventos if item[1] == "PET"]

    if not documentos_eventos_filtrados:
        print("❌ Nenhuma petição encontrada.")
        return None, None

    # === 3. Pega a petição mais recente (maior evento_id) ===
    documentos_eventos_filtrados.sort(key=lambda x: int(x[2]), reverse=True)
    documento_requerido = documentos_eventos_filtrados[0]
    evento_id = documento_requerido[2]

    # Caminho onde o PDF será salvo
    caminho_pdf = os.path.join(pasta_downloads, f"{num_processo}.pdf")

    # Localiza o elemento do documento e o link
    elemento = navegador.find_element(By.ID, documento_requerido[0])
    link = elemento.find_element(By.XPATH, ".//*[contains(@class, 'infraLinkDoc')]")

    # Limpa histórico de requests para capturar apenas a próxima
    navegador.requests.clear()

    # Clica no link para gerar a requisição ao controlador.php
    link.click()
    
    # Aguarda o carregamento do preview/divBoxPreview para garantir que o clique gerou a requisição
    WebDriverWait(navegador, 10).until(
        EC.presence_of_element_located((By.ID, "divBoxPreview"))
    )
    time.sleep(2)  # Pequeno delay extra para garantir a requisição
    # Clica no botão "open-button" para abrir o documento em nova janela/aba
    # Vai para a aba que foi aberta
    if len(navegador.window_handles) > 1:
        navegador.switch_to.window(navegador.window_handles[-1])
    try:
        timestamp_download = datetime.now().isoformat()
        navegador.find_element(By.TAG_NAME, 'body').send_keys('\t\t\n')
        time.sleep(1)
    except Exception as e:
        print("Botão 'open-button' não encontrado ou erro ao clicar:", e)
    # Fecha a nova aba/janela aberta pelo botão "open-button"
    if len(navegador.window_handles) > 1:
        # Fecha todas as abas, menos a primeira
        while len(navegador.window_handles) > 1:
            navegador.switch_to.window(navegador.window_handles[-1])
            navegador.close()
        navegador.switch_to.window(navegador.window_handles[0])
    # === 4. Procura o PDF mais recente na pasta de downloads ===

    arquivos_pdf = glob.glob(os.path.join(pasta_downloads, "*.pdf"))
    if not arquivos_pdf:
        print("❌ Nenhum PDF encontrado na pasta de downloads.")
        return evento_id, None

    # Encontra o PDF mais recente
    pdf_mais_recente = max(arquivos_pdf, key=os.path.getmtime)
    tempo_modificacao = os.path.getmtime(pdf_mais_recente)
    tempo_modificacao_dt = datetime.fromtimestamp(tempo_modificacao)
    tempo_diff = abs((tempo_modificacao_dt - datetime.fromisoformat(timestamp_download)).total_seconds())

    if tempo_diff <= 10:
        # Move o arquivo para o caminho_pdf
        os.makedirs(os.path.dirname(caminho_pdf), exist_ok=True)
        os.replace(pdf_mais_recente, caminho_pdf)
        print(f"PDF renomeado e salvo em: {caminho_pdf}")

        return evento_id, caminho_pdf
    else:
        print(f"PDF mais recente tem diferença de {tempo_diff:.2f} segundos da timestamp. Não será renomeado.")
        return evento_id, None

def extrair_texto_pdf(caminho_pdf):
    with open(caminho_pdf, 'rb') as arquivo:
        leitor = PyPDF2.PdfReader(arquivo)
        texto = ""
        for pagina in leitor.pages:
            texto += pagina.extract_text()
    return texto

def pega_texto_documento(navegador, documento):
    WebDriverWait(navegador, 20).until(
        EC.presence_of_element_located((By.ID, documento))
    )
    # 1. Localizar o elemento pelo ID
    elemento = navegador.find_element(By.ID, documento)
    # 2. Criar ActionChains para executar o mouse over
    actions = ActionChains(navegador)
    # Rolar a página para o elemento antes de mover o mouse
    navegador.execute_script("arguments[0].scrollIntoView(true); window.scrollBy(0, -150);", elemento)
    # Faz o mouseover em cima do texto link infraLinkDocumento do elemento
    link_doc = elemento.find_element(By.XPATH, ".//*[contains(@class, 'infraLinkDoc')]")
    actions.move_to_element(link_doc).perform()
    # 3. Aguardar para o hover ter efeito
    time.sleep(5)
    
    # Verifica se há uma div com a classe 'divBoxPreview' visível na página
    overlays = navegador.find_elements(By.ID, "divBoxPreview")
    visiveis = [div for div in overlays if div.is_displayed()]

    conteudo = ""
    if visiveis:
        div = overlays[0]
        # Move o foco para a div
        ActionChains(navegador).move_to_element(div).click().perform()
        # Aguarda carregar o conteúdo (ajuste o tempo se necessário)
        time.sleep(1)
        # Seleciona o texto
        conteudo = navegador.find_element(By.ID, "divBoxPreview").text
        # Clica no botão de fechar o preview, se existir
        btn_close = navegador.find_element(By.ID, "divClosePreview")
        btn_close.click()    
        time.sleep(3)

    else:
        print("Erro ao recuperar o documento")
    return conteudo

def ollama_resumo(pedido):    
    pergunta_gemma = "Considere o seguinte pedido." \
    f"{pedido}" \
    "Resuma, da maneira mais objetiva possível, o pedido. Não mencione dados pessoais, como nomes, números de documento, números de processo, valores, etc. " \
    "O resumo deve ser genérico e breve (uma frase apenas, com o mínimo de palavras possível). " \
    "Se tiver mais de um pedido, retorne uma frase para cada um." \

    resumo = ollama.chat(
        model="cnmoro/gemma3-gaia-ptbr-4b:q8_0",
        messages=[{'role': 'user', 'content': f'{pergunta_gemma}'}],    
    )

    return(resumo['message']['content'])

def verifica_tipos_de_pedidos(pedido, lista_de_pedidos):
    print("========== Verificando se é um caso de uso conhecido... ==========")
    pergunta_gemma = "Considere a seguinte lista de pedidos:" \
    f"{lista_de_pedidos}" \
    f"É possível dizer que o pedido '{pedido}' pode ser adequadamente descrito por um item dessa lista?." \
    "Se sim, retorne APENAS o texto EXATO do resumo do pedido correspondente na lista. Se não, retorne APENAS o texto 'Não'."

    resumo = ollama.chat(
        model="cnmoro/gemma3-gaia-ptbr-4b:q8_0",
        messages=[{'role': 'user', 'content': f'{pergunta_gemma}'}],    
    )

    return(resumo['message']['content'])




In [ ]:
#EXECUTA! Pega as últimas petições

# PROFILAXIA: Atualiza registros em que pet tem menos de três dígitos, tornando pet e resumo nulos, e exibe a quantidade afetada
with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM {perfil} WHERE LENGTH(pet) < 3")
    quantidade = cursor.fetchone()[0]
    cursor.execute(f"UPDATE {perfil} SET pet = NULL, resumo = NULL WHERE LENGTH(pet) < 3")
    conn.commit()
    print(f"Registros atualizados: {quantidade}")

#Gera lista de pendentes, do tipo de processo definido
#with sqlite3.connect("urcaciv.db") as conn:
    #df_pendentes = pd.read_sql_query(
        #f"SELECT * FROM {perfil} WHERE tipo = '{tipo_processo}' AND pet IS NULL", conn
        #f"SELECT * FROM {perfil} WHERE pet IS NULL", conn
    #)
#df_pendentes = df_pendentes["num_processo"].tolist()
print(f"Total de processos pendentes: {len(df_pendentes)}")

#EXECUTA! Pega as últimas petições

total_processos = len(df_pendentes)
processo_atual = 0

for processo in df_pendentes:
    # Pega o texto da área de transferência
    processo_atual += 1
    print(f"Processo {processo_atual} de {total_processos}")
    try:
        # ENTRA NO PROCESSO    
        eproc.entrar_no_processo(navegador, processo)

        # BAIXA A PETIÇÃO E A RENOMEIA
        print(f"========== Capturando petição do processo {processo} ==========")
        evento_id, doc_pdf = pega_ultima_peticao(navegador, processo)
        print(f"Evento ID: {evento_id}, Documento PDF: {doc_pdf}")

        if os.path.exists(doc_pdf):
            # LÊ A PETIÇÃO
            print(f"========== lendo a petição do processo {processo} ==========")
            texto_peticao = extrair_texto_pdf(doc_pdf)
            print(texto_peticao)

            # Atualiza o campo 'pet' no banco de dados para o processo atual
            if texto_peticao and len(texto_peticao) > 3:
                with closing(conn.cursor()) as cursor:
                    # Atualiza apenas com o texto da petição mais recente
                    cursor.execute(
                        f"UPDATE {perfil} SET pet = ? WHERE num_processo = ?",
                        (texto_peticao, processo)
                    )
                conn.commit()
            
            # Verifica se o campo 'pet' foi preenchido corretamente no banco de dados
            with closing(conn.cursor()) as cursor:
                cursor.execute(
                    f"SELECT pet FROM {perfil} WHERE num_processo = ?",
                    (processo,)
                )
                pet_db = cursor.fetchone()
                if pet_db and pet_db[0] and len(pet_db[0]) > 3 and doc_pdf and os.path.exists(doc_pdf):
                    try:
                        os.remove(doc_pdf)
                        print(f"Arquivo PDF removido: {doc_pdf}")
                    except Exception as e:
                        print(f"Erro ao remover o arquivo PDF: {e}")

                    try:
                        print(f"========== Resumindo petição do processo {processo} ==========")              
                        resumo = ollama_resumo(texto_peticao)
                        print(f"Resumo: {resumo}")
                        cursor.execute(
                        f"UPDATE {perfil} SET resumo = ? WHERE num_processo = ?",
                        (resumo, processo)
                        )
                        conn.commit()

                    except Exception as e:
                        print(f"Erro ao resumir a petição do processo {processo}: {e}")

    except:
        print(f"=========== Erro na Captura do processo {processo} ============")



Registros atualizados: 0
Total de processos pendentes: 710
Processo 1 de 710
Página do processo 5009747-60.2024.8.21.0009 carregada com sucesso.
========== Capturando petição do processo 5009747-60.2024.8.21.0009 ==========
❌ Nenhum PDF encontrado na pasta de downloads.
Evento ID: 48, Documento PDF: None
=========== Erro na Captura do processo 5009747-60.2024.8.21.0009 ============
Processo 2 de 710
=========== Erro na Captura do processo 5009541-70.2021.8.21.0132 ============
Processo 3 de 710
Página do processo 5004545-19.2024.8.21.0069 carregada com sucesso.
========== Capturando petição do processo 5004545-19.2024.8.21.0069 ==========
❌ Nenhum PDF encontrado na pasta de downloads.
Evento ID: 14, Documento PDF: None
=========== Erro na Captura do processo 5004545-19.2024.8.21.0069 ============
Processo 4 de 710


In [28]:
#Pega os processos novos para minutar, adiciona coluna de dias, pagina por 100

# Espera até o elemento estar presente e clicável
meus_localizadores = WebDriverWait(navegador, 20).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, 'i[title="Meus Localizadores"]'))
)
meus_localizadores.click()

# Aguarda algum carregamento após o clique, se necessário (exemplo: espera um painel aparecer)
# WebDriverWait(navegador, 20).until(
#     EC.visibility_of_element_located((By.ID, "id_do_painel_ou_elemento_esperado"))
# )

# Localiza o primeiro <td> que contenha "CÍVEL - MINUTAR" no texto
td_civel_minutar = WebDriverWait(navegador, 20).until(
    EC.presence_of_element_located((By.XPATH, '//td[contains(text(), "CÍVEL - MINUTAR")]'))
    #EC.presence_of_element_located((By.XPATH, '//td[contains(text(), "GABINETE - MINUTAR")]'))
    #EC.presence_of_element_located((By.XPATH, '//td[contains(text(), "GABINETE – MINUTAR")]'))    
)

# Encontra o <td> imediatamente a seguir
td_seguinte = td_civel_minutar.find_element(By.XPATH, 'following-sibling::td[1]')

# Dentro desse <td>, localiza o <a> e clica, esperando estar clicável
a_element = WebDriverWait(td_seguinte, 20).until(
    EC.element_to_be_clickable((By.TAG_NAME, 'a'))
)
a_element.click()

# Localiza e clica no label "100 processos por página"
label_100 = WebDriverWait(navegador, 20).until(
    EC.element_to_be_clickable((By.XPATH, '//label[contains(text(), "100 processos por página")]'))
)
label_100.click()

# Localiza o label com id "lbloptNdiasSituacao" e clica apenas se o checkbox estiver desmarcado
label_ndias_situacao = WebDriverWait(navegador, 20).until(
    EC.element_to_be_clickable((By.ID, "lbloptNdiasSituacao"))
)
checkbox_ndias = navegador.find_element(By.ID, "optNdiasSituacao")
if not checkbox_ndias.is_selected():
    label_ndias_situacao.click()

# Localiza o botão com id "btnConsultar" e exibe na tela
botao_consultar = navegador.find_element(By.ID, "btnConsultar")
navegador.execute_script("arguments[0].scrollIntoView();", botao_consultar)

# Tenta clicar no botão "Consultar", rolando para garantir visibilidade e tratando possíveis interceptações

try:
    botao_consultar.click()
except ElementClickInterceptedException:
    navegador.execute_script("arguments[0].scrollIntoView({block: 'center'});", botao_consultar)
    time.sleep(1)
    botao_consultar.click()

In [29]:
# Renova a tabela do perfil
with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute(f"DELETE FROM {perfil}")
    conn.commit()

# Localiza a tabela pelo id e importa as três primeiras colunas (ignorando o cabeçalho)
dados_tabela = []

pega_tabela_pagina(dados_tabela)

print(dados_tabela)

table_name = f"{perfil}"

with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()

    cursor.executemany(
        f"INSERT INTO {table_name} (num_processo, dias, tipo) VALUES (?, ?, ?)",
        dados_tabela
    )
    conn.commit()

#guarda os processos no db
with sqlite3.connect("urcaciv.db") as conn:
    df_pendentes = pd.read_sql_query(f"SELECT * FROM {perfil} WHERE tipo = '{tipo_processo}'", conn)
df_pendentes = df_pendentes["num_processo"].tolist()
print("processos a minutar guardados no banco de dados.")

[['5009747-60.2024.8.21.0009', '40', 'OUTROS PROCEDIMENTOS DE JURISDIÇÃO VOLUNTÁRIA'], ['5009541-70.2021.8.21.0132', '48', 'Guarda'], ['5004545-19.2024.8.21.0069', '4', 'PROCEDIMENTO COMUM CÍVEL'], ['5004520-40.2023.8.21.0069', '52', 'EXECUÇÃO FISCAL'], ['5004481-43.2023.8.21.0069', '21', 'EXECUÇÃO FISCAL'], ['5004463-85.2024.8.21.0069 ', '18', 'PROCEDIMENTO COMUM CÍVEL'], ['5004459-48.2024.8.21.0069', '46', 'PROCEDIMENTO COMUM CÍVEL'], ['5004451-08.2023.8.21.0069 ', '17', 'EXECUÇÃO FISCAL'], ['5004434-35.2024.8.21.0069 ', '47', 'EXECUÇÃO FISCAL'], ['5004419-03.2023.8.21.0069', '35', 'CURATELA'], ['5004401-79.2023.8.21.0069', '10', 'EXECUÇÃO FISCAL'], ['5004401-45.2024.8.21.0069', '3', 'CUMPRIMENTO PROVISÓRIO DE SENTENÇA'], ['5004397-42.2023.8.21.0069 ', '24', 'EXECUÇÃO FISCAL'], ['5004379-84.2024.8.21.0069 ', '30', 'EXECUÇÃO DE ALIMENTOS'], ['5004377-17.2024.8.21.0069', '26', 'PROCEDIMENTO COMUM CÍVEL'], ['5004374-62.2024.8.21.0069 ', '33', 'CUMPRIMENTO DE SENTENÇA DE OBRIGAÇÃO DE PRE

In [37]:
# Adiciona toda a lista de processos

# Cria a tabela, se ela não existir
with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute(f"""
        CREATE TABLE IF NOT EXISTS {perfil} (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            num_processo TEXT,
            dias INTEGER,
            tipo TEXT,
            pet TEXT,
            resumo TEXT,
            minuta TEXT,
            lote TEXT
        )
    """)
    conn.commit()

# Localiza a tabela pelo id e importa as três primeiras colunas (ignorando o cabeçalho)
dados_tabela = []

pega_tabela_pagina(dados_tabela)

print(dados_tabela)

table_name = f"{perfil}"

with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    # Insere apenas se o num_processo não existe na tabela
    for row in dados_tabela:
        num_processo = row[0]
        cursor.execute(f"SELECT 1 FROM {table_name} WHERE num_processo = ?", (num_processo,))
        if cursor.fetchone() is None:
            cursor.execute(
                f"INSERT INTO {table_name} (num_processo, dias, tipo) VALUES (?, ?, ?)",
                row
            )
    conn.commit()

#guarda os processos no db
with sqlite3.connect("urcaciv.db") as conn:
    df_pendentes = pd.read_sql_query(f"SELECT * FROM {perfil} WHERE tipo = '{tipo_processo}'", conn)
df_pendentes = df_pendentes["num_processo"].tolist()
print("processos a minutar guardados no banco de dados.")

[['5000011-18.2013.8.21.0069', '7', 'CUMPRIMENTO DE SENTENÇA CONTRA A FAZENDA PÚBLICA'], ['5000011-03.2022.8.21.0069', '7', 'EXECUÇÃO DE TÍTULO EXTRAJUDICIAL'], ['5000010-72.2009.8.21.0069', '40', 'EXECUÇÃO FISCAL'], ['5000009-24.2008.8.21.0069', '4', 'EXECUÇÃO FISCAL'], ['5000007-97.2021.8.21.0069 ', '56', 'EXECUÇÃO FISCAL'], ['5000005-84.2008.8.21.0069 ', '4', 'EXECUÇÃO FISCAL'], ['5000005-21.2007.8.21.0069 ', '12', 'PROCEDIMENTO COMUM CÍVEL'], ['5000003-90.2003.8.21.0069', '19', 'CUMPRIMENTO DE SENTENÇA'], ['5000003-80.2009.8.21.0069', '17', 'EXECUÇÃO FISCAL'], ['5000001-81.2007.8.21.0069', '3', 'CUMPRIMENTO DE SENTENÇA']]
processos a minutar guardados no banco de dados.


In [38]:
#Gera lista de pendentes, do tipo de processo definido

with sqlite3.connect("urcaciv.db") as conn:
    df_pendentes = pd.read_sql_query(
        #f"SELECT * FROM {perfil} WHERE tipo = '{tipo_processo}' AND pet IS NULL", conn
        f"SELECT * FROM {perfil} WHERE pet IS NULL", conn
    )
df_pendentes = df_pendentes["num_processo"].tolist()

print(f"Total de processos pendentes: {len(df_pendentes)}")


Total de processos pendentes: 710


In [4]:
# Atualiza registros em que pet tem menos de três dígitos, tornando pet e resumo nulos, e exibe a quantidade afetada
with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM {perfil} WHERE LENGTH(pet) < 3")
    quantidade = cursor.fetchone()[0]
    cursor.execute(f"UPDATE {perfil} SET pet = NULL, resumo = NULL WHERE LENGTH(pet) < 3")
    conn.commit()
    print(f"Registros atualizados: {quantidade}")

Registros atualizados: 153


In [5]:
# Apaga todas as petições que forem "0"
with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute(f"UPDATE {perfil} SET pet = NULL WHERE pet = '0'")
    conn.commit()
    print("Todas as petições com valor '0' foram apagadas.")

Todas as petições com valor '0' foram apagadas.


In [5]:
# Reseta os repetidos
with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute(f"""
        SELECT num_processo, pet
        FROM {perfil}
        WHERE pet IS NOT NULL
        ORDER BY id
    """)
    registros = cursor.fetchall()

    repetidos = 0
    for i in range(1, len(registros)):
        if registros[i][1] == registros[i-1][1]:
            cursor.execute(
                f"UPDATE {perfil} SET pet = NULL, resumo = NULL WHERE num_processo = ?",
                (registros[i][0],)
            )
    conn.commit()

    print(f"Total de registros com pet igual ao registro anterior: {repetidos}")

    cursor.execute(f"SELECT num_processo FROM {perfil} WHERE pet IS NULL")
    df_pendentes = [row[0] for row in cursor.fetchall()]
    print(f"Total de processos pendentes: {len(df_pendentes)}")


Total de registros com pet igual ao registro anterior: 0
Total de processos pendentes: 1746


In [ ]:
# Gera resumo das petições do db
import ollama
import sqlite3

with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute(f"SELECT num_processo, pet FROM {perfil} WHERE pet IS NOT NULL AND resumo IS NULL")
    processos = cursor.fetchall()
    processos_total = len(processos)
    processo_atual = 0

    for num_processo, texto_pet in processos:
        processo_atual += 1
        print(f"========== Resumindo petição do processo {num_processo} ({processo_atual} de {processos_total}) ==========")
        resumo = ollama_resumo(texto_pet)
        print(f"Resumo: {resumo}")
        cursor.execute(
            f"UPDATE {perfil} SET resumo = ? WHERE num_processo = ?",
            (resumo, num_processo)
        )
        conn.commit()
    

========== Resumindo petição do processo 5032339-44.2024.8.21.0027 (1 de 336) ==========
Resumo: O Procurador do Estado solicita o acolhimento da petição para o fim da penhora do veículo, com base na legislação e jurisprudência aplicáveis.
========== Resumindo petição do processo 5032324-12.2023.8.21.0027 (2 de 336) ==========
Resumo: O Procurador do Estado solicita o acolhimento da petição para o fim da penhora do veículo, com base na legislação e jurisprudência aplicáveis.
========== Resumindo petição do processo 5032324-12.2023.8.21.0027 (2 de 336) ==========
Resumo: O Município de Santa Maria requer o levantamento das restrições patrimoniais e a exclusão do nome da parte executada dos cadastros de restrição de crédito.

========== Resumindo petição do processo 5032053-71.2021.8.21.0027  (3 de 336) ==========
Resumo: O Município de Santa Maria requer o levantamento das restrições patrimoniais e a exclusão do nome da parte executada dos cadastros de restrição de crédito.

========== 

In [ ]:
#EXECUTA! Pega e resume as últimas petições
def analisa_processo(navegador, processo):
    navegador.switch_to.default_content()
    eproc.entrar_no_processo(navegador, processo)

    with sqlite3.connect("urcaciv.db") as conn:
        evento_id, texto_peticao = pega_ultima_peticao(navegador)

        print(f"========== TEXTO DA PETIÇÃO ==========")
        print(texto_peticao)


        with closing(conn.cursor()) as cursor:
            cursor.execute(
                f"UPDATE {perfil} SET pet = ? WHERE num_processo = ?",    
                (texto_peticao, processo)
            )
            conn.commit()
        
        resumo_ollama = ollama_resumo(texto_peticao)
        print(f"========== RESUMO DA PETIÇÃO ==========")
        print(resumo_ollama)

        with closing(conn.cursor()) as cursor:
            cursor.execute(
                f"UPDATE {perfil} SET resumo = ? WHERE num_processo = ?",    
                (resumo_ollama, processo)
            )
            conn.commit()
        #verifica os tipos de pedidos conhecidos
        with sqlite3.connect("movimentos.db") as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT id, resumo FROM pedidos")
            tipos_pedidos = cursor.fetchall()
            tipos_pedidos = "\n".join([f"ID: {id}, Resumo: {resumo}" for id, resumo in tipos_pedidos])
        
        resultado = verifica_tipos_de_pedidos(resumo_ollama, tipos_pedidos)
        print(resultado)
        resultado_limpo = re.sub(r'[\r\n]+', ' ', resultado).strip()
        resultado = re.sub(r'[^\w\s.,;:!?-]', '', resultado_limpo)       
        
        # Limpa o resultado de retornos e caracteres especiais




for processo in df_pendentes:
    analisa_processo(navegador, processo)



In [23]:
#Insere os Lembretes que estão em lembretes_to_do


def insere_lembrete(navegador, texto, validade):

    data_atual = datetime.now().strftime("%d/%m/%Y")
    data_atual = data_atual + " 00:00"
    data_fim = (datetime.now() + timedelta(days=validade)).strftime("%d/%m/%Y")
    data_fim = data_fim + " 00:00"
    texto_lembrete = "Solicita-se a desconsideração do valor de execução irrisório e o prosseguimento do processo."
    lembrete = f"{data_atual} - {texto_lembrete}"


    # Localiza o campo de lembrete e insere o texto    
    navegador.switch_to.default_content()
    novo_btn = navegador.find_element(By.LINK_TEXT, "Novo")

    time.sleep(0.5)
    novo_btn.click()

    # Aguarda até o frame estar visível antes de trocar para ele
    WebDriverWait(navegador, 10).until(
        EC.frame_to_be_available_and_switch_to_it(1)
    )

    # Aguarda o carregamento completo do frame antes de prosseguir
    WebDriverWait(navegador, 20).until(
        EC.presence_of_element_located((By.ID, "txaDescricao"))
    )
    navegador.find_element(By.ID, "txaDescricao").click()
    navegador.find_element(By.ID, "txaDescricao").send_keys(texto)
    navegador.find_element(By.CSS_SELECTOR, "td:nth-child(2) > .infraRadio").click()

    # Dá scroll para o campo ficar o mais alto possível na tela
    campo_validade = navegador.find_element(By.ID, "chkInformaValidade")
    navegador.execute_script("arguments[0].scrollIntoView(true); window.scrollBy(0, -150);", campo_validade)
    campo_validade.click()

    navegador.find_element(By.ID, "txtDataInicio").clear()
    navegador.find_element(By.ID, "txtDataInicio").send_keys(data_atual)

    navegador.find_element(By.ID, "txtDataTermino").clear()
    navegador.find_element(By.ID, "txtDataTermino").send_keys(data_fim)


    # Dá scroll para o botão de salvar ficar visível na tela antes de clicar
    btn_salvar = navegador.find_element(By.CSS_SELECTOR, "#divInfraBarraComandosInferior > #sbmSalvar")
    navegador.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn_salvar)
    time.sleep(0.5)
    btn_salvar.click()
    navegador.switch_to.default_content()
    time.sleep(2)

validade = 6  # Defina a validade do lembrete em dias

with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM lembretes_to_do WHERE dt_lembrete IS NULL")
    lembretes_pendentes = cursor.fetchall()
    total_pendentes = len(lembretes_pendentes)
    print(f"Total de lembretes pendentes: {total_pendentes}")
    pendente_atual = 0

    for lembrete in lembretes_pendentes:
        pendente_atual += 1
        num_processo = lembrete[1].replace(" ", "")
        texto_lembrete = "URCA - " + lembrete[2]
        print(f"Processo {pendente_atual} de {total_pendentes}: {num_processo}. Lembrete: {texto_lembrete}")                
        
        #EXECUTA
        navegador.switch_to.default_content()
        eproc.entrar_no_processo(navegador, num_processo)
        time.sleep(1)        
        insere_lembrete(navegador, texto_lembrete, validade)        
        
        timestamp = datetime.now().strftime("%d/%m/%Y")
        cursor.execute(
            "UPDATE lembretes_to_do SET dt_lembrete = ? WHERE num_processo = ? AND unidade = ?",
            (timestamp, num_processo, lembrete[0])
        )
        conn.commit()
        


Total de lembretes pendentes: 482
Processo 1 de 482: 5041886-79.2022.8.21.0027. Lembrete: URCA - Citação edital
Página do processo 5041886-79.2022.8.21.0027 carregada com sucesso.
Processo 2 de 482: 5041365-37.2022.8.21.0027. Lembrete: URCA - Citação edital
Página do processo 5041365-37.2022.8.21.0027 carregada com sucesso.
Processo 3 de 482: 5040902-95.2022.8.21.0027. Lembrete: URCA - Citação edital
Página do processo 5040902-95.2022.8.21.0027 carregada com sucesso.
Processo 4 de 482: 5040891-66.2022.8.21.0027. Lembrete: URCA - Citação edital
Página do processo 5040891-66.2022.8.21.0027 carregada com sucesso.
Processo 5 de 482: 5040804-13.2022.8.21.0027. Lembrete: URCA - Citação edital
Página do processo 5040804-13.2022.8.21.0027 carregada com sucesso.
Processo 6 de 482: 5040801-58.2022.8.21.0027. Lembrete: URCA - Citação edital
Página do processo 5040801-58.2022.8.21.0027 carregada com sucesso.
Processo 7 de 482: 5040570-94.2023.8.21.0027. Lembrete: URCA - Citação edital
Página do pr

In [ ]:
#Apaga os lembretes do tipo selecionado

import sqlite3
import time 

tipo = 'SUSPENSÃO PARCELAMENTO'
with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM lembretes_to_do WHERE tipo_pedido = ?", (tipo,))
    processos_renajud = cursor.fetchall()

print(f"Total de processos com tipo_pedido '{tipo}': {len(processos_renajud)}")
for processo in processos_renajud:
    num_processo = processo[1]
    eproc.entrar_no_processo(navegador, num_processo)
    time.sleep(1)        
    try:
        div_conteudo_lembretes = navegador.find_element(By.ID, "divConteudoLembretes")
        div_lembretes = div_conteudo_lembretes.find_elements(By.CLASS_NAME, "divLembrete")
        if div_lembretes:
            div_lembrete = div_lembretes[0]
            des_lembretes = div_lembrete.find_elements(By.CLASS_NAME, "desLembrete")
            if des_lembretes:
                texto_lembrete = des_lembretes[0].text
                if "URCA - " in texto_lembrete:
                    eproc.apaga_ultimo_lembrete(navegador)
    except:
        print(f"Erro ao apagar lembrete do processo {num_processo}. Verifique se o lembrete existe.")


In [24]:
from datetime import datetime

data_hoje = datetime.now().strftime("%d/%m/%Y")
print(f"Data de hoje: {data_hoje}")
with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT tipo_pedido, COUNT(*) FROM lembretes_to_do WHERE dt_lembrete LIKE ? GROUP BY tipo_pedido", (f"{data_hoje}%",))

    resultados = cursor.fetchall()    
    print("========== Resultados dos lembretes por tipo de pedido ==========")
    for tipo_pedido, quantidade in resultados:
        print(f'Texto do lembrete: "URCA - {tipo_pedido}" - Quantidade: {quantidade} processos. Validade dos lembretes : {validade} dias.')

Data de hoje: 18/08/2025
========== Resultados dos lembretes por tipo de pedido ==========
Texto do lembrete: "URCA - Citação edital" - Quantidade: 65 processos. Validade dos lembretes : 6 dias.
Texto do lembrete: "URCA - Intimação" - Quantidade: 2 processos. Validade dos lembretes : 6 dias.
Texto do lembrete: "URCA - Intimação edital" - Quantidade: 16 processos. Validade dos lembretes : 6 dias.
Texto do lembrete: "URCA - RENAJUD" - Quantidade: 106 processos. Validade dos lembretes : 6 dias.
Texto do lembrete: "URCA - Suspensão Parcelamento " - Quantidade: 120 processos. Validade dos lembretes : 6 dias.
Texto do lembrete: "URCA - extinção quitação" - Quantidade: 156 processos. Validade dos lembretes : 6 dias.
Texto do lembrete: "URCA - infojud bens" - Quantidade: 6 processos. Validade dos lembretes : 6 dias.
Texto do lembrete: "URCA - infojud endereço" - Quantidade: 11 processos. Validade dos lembretes : 6 dias.


In [7]:
# Mostra os tipos de pedidos conhecidos'
with sqlite3.connect("movimentos.db") as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT id, resumo FROM pedidos")
    tipos_pedidos = cursor.fetchall()
    tipos_pedidos = "\n".join([f"ID: {id}, Resumo: {resumo}" for id, resumo in tipos_pedidos])
    print(tipos_pedidos)

ID: 1, Resumo: Pedido de desistência do processo
ID: 2, Resumo: Informação de que a parte está ciente.
ID: 3, Resumo: Solicita a inclusão da parte executada em cadastros de inadimplentes para fins de cobrança de dívida tributária.
ID: 4, Resumo: Pedido de desconsideração do valor irrisório bloqueado e regular prosseguimento do processo.
ID: 5, Resumo: O município solicita um prazo estendido para realizar o protesto do título, visando à extinção da execução fiscal de baixo valor.
ID: 6, Resumo: Solicita-se a desconsideração do valor irrisório bloqueado e regular prosseguimento do processo.
ID: 7, Resumo: Solicita-se a penhora online via Sisbajud, modalidade reiterada, para identificar e bloquear valores em contas da executada.


In [10]:

eproc.apaga_ultimo_lembrete(navegador)

In [33]:
print(verifica_tipos_de_pedidos(resumo, tipos_pedidos))

========== Verificando se é um caso de uso conhecido... ==========
Sim
4



In [ ]:
#EXECUTA! Pega e resume as últimas petições
for processo in df_pendentes:
    eproc.entrar_no_processo(navegador, processo)

    with sqlite3.connect("urcaciv.db") as conn:
        texto_peticao = pega_ultima_peticao(navegador)

        print("========== RESUMO ==========")
        print(texto_peticao)


        with closing(conn.cursor()) as cursor:
            cursor.execute(
                f"UPDATE {perfil} SET pet = ? WHERE num_processo = ?",    
                (texto_peticao, processo)
            )
            conn.commit()
        
        resumo_ollama = ollama_resumo(texto_peticao)
        print("========== RESUMO ==========")
        print(resumo_ollama)
        print("============================")

        with closing(conn.cursor()) as cursor:
            cursor.execute(
                f"UPDATE {perfil} SET resumo = ? WHERE num_processo = ?",    
                (resumo_ollama, processo)
            )
            conn.commit()



Página do processo 5004481-43.2023.8.21.0069 carregada com sucesso.
========== RESUMO ==========

ESTADO DO RIO GRANDE DO SUL
PREFEITURA MUNICIPAL DE BARRA FUNDA
Av. 24 de Março, 735 – Centro – Fone (54) 99655-8503 – Cep 99.585-000 – Barra Funda - RS 1
AO JUÍZO DA VARA JUDICIAL DA COMARCA DE SARANDI/RS.
EXECUÇÃO FISCAL Nº 5004481-43.2023.8.21.0069
MUNICÍPIO DE BARRA FUNDA, pessoa Jurídica de Direito Público interno, por
meio da assessora jurídica que subscreve, vem respeitosamente perante Vossa
Excelência, nos autos do processo que move em face de SUZANA ANDRADE, requerer
o que segue:
Considerando o despacho proferido sobre a aplicação do Tema 1184 de
Repercussão Geral do STF e da Resolução n. 547 do CNJ, que dispõem sobre a
extinção de execuções fiscais de baixo valor, e tendo em vista que o protesto do
título é uma condição prevista para o ajuizamento da execução fiscal e que a
Fazenda Pública não teve a oportunidade de realizá-lo, requer-se a concessão de
prazo de 180 (cento e oiten

InvalidSessionIdException: Message: invalid session id
Stacktrace:
	GetHandleVerifier [0x0x7ff7221d6f75+76917]
	GetHandleVerifier [0x0x7ff7221d6fd0+77008]
	(No symbol) [0x0x7ff721f89c1c]
	(No symbol) [0x0x7ff721fd055f]
	(No symbol) [0x0x7ff722008332]
	(No symbol) [0x0x7ff722002e53]
	(No symbol) [0x0x7ff722001f19]
	(No symbol) [0x0x7ff721f54b05]
	GetHandleVerifier [0x0x7ff7224ad2ad+3051437]
	GetHandleVerifier [0x0x7ff7224a7903+3028483]
	GetHandleVerifier [0x0x7ff7224c589d+3151261]
	GetHandleVerifier [0x0x7ff7221f183e+185662]
	GetHandleVerifier [0x0x7ff7221f96ff+218111]
	(No symbol) [0x0x7ff721f53b00]
	GetHandleVerifier [0x0x7ff7225c5f18+4201496]
	BaseThreadInitThunk [0x0x7ffda032e8d7+23]
	RtlUserThreadStart [0x0x7ffda0a7c34c+44]


In [7]:
#Cria um digesto:
with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute(f"SELECT num_processo, resumo FROM {perfil} WHERE resumo IS NOT NULL")
    resultados = cursor.fetchall()
    for num_processo, resumo in resultados:
        print(f"Processo: {num_processo}")
        print(f"Resumo: {resumo}")
        print("-" * 40)

Processo: 5004481-43.2023.8.21.0069
Resumo: O município solicita um prazo estendido para realizar o protesto do título, visando à extinção da execução fiscal de baixo valor, em conformidade com as determinações do STF e do CNJ.
----------------------------------------
Processo: 5004480-58.2023.8.21.0069
Resumo: O município solicita um prazo estendido para realizar o protesto do título, visando à extinção da execução fiscal e ao cumprimento do devido processo legal.
----------------------------------------
Processo: 5004479-73.2023.8.21.0069
Resumo: Solicita-se prorrogação do prazo para realizar o protesto do título, sob a alegação de que o prazo inicial é insuficiente para a devida condução do processo de extinção da execução fiscal.
----------------------------------------
Processo: 5004411-26.2023.8.21.0069 
Resumo: O Município solicita um prazo estendido para realizar o protesto do título, visando à extinção da execução fiscal.
----------------------------------------
Processo: 5004

In [1]:
# Atualiza registros em que pet tem menos de três dígitos, tornando pet e resumo nulos, e exibe a quantidade afetada
with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM {perfil} WHERE LENGTH(pet) < 3")
    quantidade = cursor.fetchone()[0]
    cursor.execute(f"UPDATE {perfil} SET pet = NULL, resumo = NULL WHERE LENGTH(pet) < 3")
    conn.commit()
    print(f"Registros atualizados: {quantidade}")

NameError: name 'sqlite3' is not defined